# VAE Notebook

Use a VAE to enable inpainting of the galaxy photos. First we just want to be able to encode and decode the images

In [5]:
import matplotlib as mpl
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd 
import random 

import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn
import torch.nn.functional as F
import torch.optim as optim

from umap import UMAP

from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
import itertools


import h5py
from tensorflow.keras import utils

Assume images have the format:
- size 256x256
- array shape (N, C, H, W) (i.e. number, channel, height, width) with is default for PyTorch.
- assume float32, scaled to [0,1]
- one centered galaxy per image
- 3 channels/colors (g,r,z)

Before being finalized, I'll test out the framework for the VAE on slightly smaller images (maybe reduce to 64x64)

First find the galaxy data:

In [6]:
from pathlib import Path
import urllib.request

filename = "Galaxy10_DECals.h5"
url = "https://www.astro.utoronto.ca/~bovy/Galaxy10/Galaxy10_DECals.h5"  # from the astroNN docs page

# Same search as the data_processing notebook: look in data/ here and in parent folders
project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / filename).exists()),
    None,
)

if project_root is None:  # not found anywhere, so download it (~2.5 GB)
    project_root = Path.cwd()
    (project_root / "data").mkdir(exist_ok=True)
    print("Downloading Galaxy10 DECaLS...")
    urllib.request.urlretrieve(url, project_root / "data" / filename)

data_path = project_root / "data" / filename
print("Using:", data_path)

Using: /Users/hjreno2003/Desktop/Missouri/Projects/ML_finalproj/data/Galaxy10_DECals.h5


split data into training, validation, and test sets with dataloaders

In [ ]:
# ---- config ----
IMG_SIZE   = 64      # use 64 to start (faster debugging), try 128 or 256 later
BATCH_SIZE = 128
SEED       = 42

device = torch.device(
    "mps" if torch.backends.mps.is_available() #Apple Silicon GPU
    else "cuda" if torch.cuda.is_available() #NVIDIA GPU
    else "cpu"
)
print("Device:", device)

# create a loader that reads the Galaxy10 data from the h5 file in chunks, resizes to img_size, and returns uint8 images and int64 labels
def load_galaxy10(path, img_size=64, chunk=1000):
    """Return uint8 images (N, 3, img_size, img_size) and int64 labels (N,)."""
    with h5py.File(path, "r") as f:
        n = f["images"].shape[0]
        labels = torch.from_numpy(f["ans"][:].astype(np.int64)) # get labels as int64, convert to torch tensor
        out = torch.empty((n, 3, img_size, img_size), dtype=torch.uint8)

        for i in range(0, n, chunk):  # chunked so we never hold all 256x256 images as float
            x = torch.from_numpy(f["images"][i:i + chunk])      # (n, H, W, C) uint8
            x = x.permute(0, 3, 1, 2).float()                   # -> (n, C, H, W)
            if x.shape[-1] != img_size:
                x = F.interpolate(x, size=(img_size, img_size), mode="area")
            out[i:i + chunk] = x.round().clamp(0, 255).to(torch.uint8)
    return out, labels


class GalaxyDataset(Dataset):
    """Keeps images as uint8 in memory (small) and converts to float32 in [0,1] on access."""
    def __init__(self, images_u8, labels):
        self.images = images_u8
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        return self.images[i].float() / 255.0, self.labels[i]


images_u8, labels = load_galaxy10(data_path, img_size=IMG_SIZE)
dataset = GalaxyDataset(images_u8, labels)
print("Loaded:", tuple(images_u8.shape), images_u8.dtype)

# Fixed 80/10/10 split (same seed -> same split every run)
n = len(dataset)
n_train, n_val = int(0.8 * n), int(0.1 * n)
n_test = n - n_train - n_val
train_ds, val_ds, test_ds = random_split(
    dataset, [n_train, n_val, n_test], generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"train/val/test = {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")

Device: mps
Loaded: (17736, 3, 64, 64) torch.uint8
train/val/test = 14188/1773/1775
